## 1. Install Libraries

In [9]:
!pip install -q scikit-learn
!pip install -q nltk
!pip install -q joblib

## 2. Imports

In [10]:
import os
import gc
import time
import warnings

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")

## 3. Project Paths

In [11]:
PROJECT_PATH="/content/drive/MyDrive/FinSight_AI"

DATA_PATH=os.path.join(
    PROJECT_PATH,
    "data"
)

PROCESSED_PATH=os.path.join(
    DATA_PATH,
    "processed"
)

CLUSTER_PATH=os.path.join(
    DATA_PATH,
    "clusters"
)

CLUSTER_BATCH_PATH=os.path.join(
    CLUSTER_PATH,
    "cluster_batches"
)

TOPIC_PATH=os.path.join(
    DATA_PATH,
    "topics"
)

REPORT_PATH=os.path.join(
    PROJECT_PATH,
    "reports"
)

os.makedirs(
    TOPIC_PATH,
    exist_ok=True
)

## 4. Load Cluster Manifest

In [12]:
cluster_manifest=pd.read_parquet(

    os.path.join(

        CLUSTER_PATH,

        "cluster_manifest.parquet"

    )

)

cluster_manifest.head()

,batch_number,rows,cluster_file
0,433,5000,cluster_batch_0433.parquet
1,434,5000,cluster_batch_0434.parquet
2,435,5000,cluster_batch_0435.parquet
3,436,5000,cluster_batch_0436.parquet
4,437,5000,cluster_batch_0437.parquet


## 5. Load Financial News

In [13]:
financial_news=pd.read_parquet(

    os.path.join(

        PROCESSED_PATH,

        "financial_news_clean.parquet"

    ),

    columns=[

        "news_id",

        "headline"

    ]

)

print(financial_news.shape)

financial_news.head()

(3215296, 2)


,news_id,headline
0,1,Stocks That Hit 52-Week Highs On Friday
1,2,Stocks That Hit 52-Week Highs On Wednesday
2,3,71 Biggest Movers From Friday
3,4,46 Stocks Moving In Friday's Mid-Day Session
4,5,B of A Securities Maintains Neutral on Agilent...


## 6. Merge All Cluster Batches

In [14]:
import os
import gc
import time

MERGED_BATCH_PATH = os.path.join(
    TOPIC_PATH,
    "clustered_batches"
)

os.makedirs(
    MERGED_BATCH_PATH,
    exist_ok=True
)

cluster_files = sorted(
    os.listdir(CLUSTER_BATCH_PATH)
)

overall_start = time.time()

processed_rows = 0

print("="*70)
print("Building Clustered News Batches")
print("="*70)

for batch_idx, file in enumerate(

    tqdm(
        cluster_files,
        desc="Processing Batches"
    )

):

    output_file = os.path.join(

        MERGED_BATCH_PATH,

        f"merged_batch_{batch_idx+1:04d}.parquet"

    )

    if os.path.exists(output_file):

        continue

    cluster_df = pd.read_parquet(

        os.path.join(
            CLUSTER_BATCH_PATH,
            file
        ),

        columns=[
            "news_id",
            "cluster",
            "confidence",
            "distance_to_centroid"
        ]

    )

    merged_df = cluster_df.merge(

        financial_news,

        on="news_id",

        how="left"

    )

    merged_df.to_parquet(

        output_file,

        index=False

    )

    processed_rows += len(merged_df)

    elapsed = time.time() - overall_start

    avg_batch = elapsed / (batch_idx + 1)

    eta = (

        len(cluster_files)

        - batch_idx

        - 1

    ) * avg_batch / 60

    if (

        (batch_idx + 1) % 25 == 0

        or

        batch_idx == len(cluster_files) - 1

    ):

        print()

        print("="*70)

        print(f"Batch : {batch_idx+1}/{len(cluster_files)}")

        print(f"Rows Processed : {processed_rows:,}")

        print(f"ETA : {eta:.2f} minutes")

    del cluster_df
    del merged_df

    gc.collect()

print()

print("="*70)
print("ALL BATCHES CREATED")
print("="*70)

print(f"Total Rows : {processed_rows:,}")

print(f"Merged Batches : {len(cluster_files)}")

Building Clustered News Batches


Processing Batches:   0%|          | 0/644 [00:00<?, ?it/s]


Batch : 25/644
Rows Processed : 125,000
ETA : 7.11 minutes

Batch : 50/644
Rows Processed : 250,000
ETA : 5.02 minutes

Batch : 75/644
Rows Processed : 375,000
ETA : 4.19 minutes

Batch : 100/644
Rows Processed : 500,000
ETA : 3.70 minutes

Batch : 125/644
Rows Processed : 625,000
ETA : 3.39 minutes

Batch : 150/644
Rows Processed : 750,000
ETA : 3.09 minutes

Batch : 175/644
Rows Processed : 875,000
ETA : 2.88 minutes

Batch : 200/644
Rows Processed : 1,000,000
ETA : 2.71 minutes

Batch : 225/644
Rows Processed : 1,125,000
ETA : 2.51 minutes

Batch : 250/644
Rows Processed : 1,250,000
ETA : 2.33 minutes

Batch : 275/644
Rows Processed : 1,375,000
ETA : 2.17 minutes

Batch : 300/644
Rows Processed : 1,500,000
ETA : 2.00 minutes

Batch : 325/644
Rows Processed : 1,625,000
ETA : 1.85 minutes

Batch : 350/644
Rows Processed : 1,750,000
ETA : 1.69 minutes

Batch : 375/644
Rows Processed : 1,875,000
ETA : 1.54 minutes

Batch : 400/644
Rows Processed : 2,000,000
ETA : 1.39 minutes

Batch : 

## 7. Cluster Statistics

In [16]:
cluster_statistics = pd.read_parquet(

    os.path.join(

        CLUSTER_PATH,

        "cluster_statistics.parquet"

    )

)

cluster_statistics.head()

,cluster,headlines,avg_confidence,representative_distance
0,0,1647,0.848314,1.160776e-01
1,1,198,0.996909,8.560065e-08
2,2,790,0.998421,4.750784e-06
3,3,56,0.643269,2.107342e-08
4,4,439,0.716785,2.302391e-01


In [17]:
cluster_summary = (

    clustered_news

    .groupby("cluster")

    .agg(

        headline_count=("news_id","count"),

        avg_confidence=("confidence","mean"),

        avg_distance=("distance_to_centroid","mean")

    )

    .reset_index()

)

cluster_summary = cluster_summary.merge(

    cluster_statistics,

    on="cluster",

    how="left"

)

cluster_summary.head()

,cluster,headline_count,avg_confidence_x,avg_distance,headlines,avg_confidence_y,representative_distance
0,0,1647,0.847356,0.181571,1647,0.848314,1.160776e-01
1,1,198,0.996189,0.004489,198,0.996909,8.560065e-08
2,2,790,0.998277,0.002363,790,0.998421,4.750784e-06
3,3,56,0.608159,0.671938,56,0.643269,2.107342e-08
4,4,439,0.728438,0.383847,439,0.716785,2.302391e-01


## 8. Build Documents for Each Cluster

In [18]:
cluster_documents = (

    clustered_news

    .groupby("cluster")["headline"]

    .apply(

        lambda x: " ".join(x.astype(str))

    )

    .reset_index()

)

print(cluster_documents.shape)

cluster_documents.head()

(2913, 2)


,cluster,headline
0,0,Watch These 8 Huge Call Purchases In Thursday ...
1,1,Healthcare ratings roundup Healthcare ratings ...
2,2,Morning Market Losers Morning Market Losers Mo...
3,3,How LikeFolio Predicted Good News For Bebe Sto...
4,4,Shares of several basic material companies are...


## 9. TF-IDF

In [19]:
vectorizer = TfidfVectorizer(

    stop_words="english",

    max_features=5000,

    ngram_range=(1,2),

    min_df=2

)

tfidf_matrix = vectorizer.fit_transform(

    cluster_documents["headline"]

)

feature_names = np.array(

    vectorizer.get_feature_names_out()

)

print(tfidf_matrix.shape)

(2913, 5000)


## 10. Extract Top Keywords

In [20]:
TOP_K = 15

keywords = []

print("=" * 70)
print("Extracting Top Keywords")
print("=" * 70)

for row in tqdm(

    range(

        tfidf_matrix.shape[0]

    ),

    desc="Processing Clusters"

):

    scores = tfidf_matrix[row].toarray().ravel()

    top_idx = np.argsort(

        scores

    )[-TOP_K:][::-1]

    top_words = feature_names[

        top_idx

    ]

    keywords.append(

        top_words.tolist()

    )

print()

print("=" * 70)
print("Keyword Extraction Completed")
print("=" * 70)

print(f"Clusters Processed : {len(keywords):,}")

Extracting Top Keywords


Processing Clusters:   0%|          | 0/2913 [00:00<?, ?it/s]


Keyword Extraction Completed
Clusters Processed : 2,913


## 11. Build Topic DataFrame

In [21]:
topics_df = cluster_documents.copy()

topics_df["keywords"] = keywords

topics_df.head()

,cluster,headline,keywords
0,0,Watch These 8 Huge Call Purchases In Thursday ...,"[huge purchases, trade watch, watch huge, purc..."
1,1,Healthcare ratings roundup Healthcare ratings ...,"[roundup, ratings, healthcare, ratings round, ..."
2,2,Morning Market Losers Morning Market Losers Mo...,"[losers morning, market losers, morning market..."
3,3,How LikeFolio Predicted Good News For Bebe Sto...,"[stores, tale tape, tape, tale, stock, focus s..."
4,4,Shares of several basic material companies are...,"[trading lower, companies trading, basic mater..."


## 12. Generate Topic Names

In [31]:
GENERIC_WORDS = {
    "stock", "stocks", "share", "shares",
    "company", "companies",
    "market", "markets",
    "report", "reports", "reported",
    "financial", "news", "update", "updates",
    "quarter", "quarterly",
    "analyst", "analysts",
    "inc", "corp", "corporation",
    "holding", "holdings",
    "group", "results", "transcript",
    "conference", "call", "llc",
    "earnings", "today", "week",
    "month", "year"
}

def generate_topic(keywords, headline, max_words=3):

    words = []
    seen = set()

    for phrase in keywords:

        for word in phrase.lower().split():

            if word in GENERIC_WORDS:
                continue

            if len(word) < 3:
                continue

            if word in seen:
                continue

            seen.add(word)
            words.append(word.title())

            if len(words) == max_words:
                return " ".join(words)

    # fallback
    return " ".join(headline.split()[:4])

In [32]:
topics_df["topic_name"] = topics_df.apply(

    lambda row:

    generate_topic(

        row["keywords"],

        row["headline"]

    ),

    axis=1

)

topics_df.head()

,cluster,topic_name,headline,keywords,headline_count,avg_confidence,avg_distance
0,1521,Ceo 2016 2018,From Applied Materials Q2 Earnings Conference ...,"[earnings transcript, results earnings, transc...",36014,0.588888,0.521184
1,310,Declares Dividend Fund,Dundee Corporation Declares Quarterly First Pr...,"[declares, dividend, fund declares, trust decl...",26158,0.582308,0.537981
2,391,Buys Sells Management,"Greenlight Capital Buys Yahoo!, Trims Apple, S...","[buys, llc buys, sells, corp, llc, management,...",25802,0.580003,0.537210
3,2996,Estimates Revenues Beats,Agilent Q4 Profit Beats Views Agilent Beats Q3...,"[earnings, estimates, revenues, earnings reven...",23335,0.571469,0.583278
4,1604,Buy Value Investors,Why Agilent Technologies (A) Stock Might be a ...,"[stock, buy, value, value investors, investors...",21029,0.570697,0.581782


## 13. Merge Cluster Statistics

In [33]:
topics_df = topics_df.merge(

    cluster_statistics,

    on="cluster",

    how="left"

)

topics_df.head()

,cluster,topic_name,headline,keywords,headline_count,avg_confidence_x,avg_distance,headlines,avg_confidence_y,representative_distance
0,1521,Ceo 2016 2018,From Applied Materials Q2 Earnings Conference ...,"[earnings transcript, results earnings, transc...",36014,0.588888,0.521184,36014,0.588888,0.521184
1,310,Declares Dividend Fund,Dundee Corporation Declares Quarterly First Pr...,"[declares, dividend, fund declares, trust decl...",26158,0.582308,0.537981,26158,0.582308,0.537981
2,391,Buys Sells Management,"Greenlight Capital Buys Yahoo!, Trims Apple, S...","[buys, llc buys, sells, corp, llc, management,...",25802,0.580003,0.537210,25802,0.580003,0.537210
3,2996,Estimates Revenues Beats,Agilent Q4 Profit Beats Views Agilent Beats Q3...,"[earnings, estimates, revenues, earnings reven...",23335,0.571469,0.583278,23335,0.571469,0.583278
4,1604,Buy Value Investors,Why Agilent Technologies (A) Stock Might be a ...,"[stock, buy, value, value investors, investors...",21029,0.570697,0.581782,21029,0.570697,0.581782


## 14. Reorder Columns

In [34]:
topics_df.columns

Index(['cluster', 'topic_name', 'headline', 'keywords', 'headline_count',
       'avg_confidence_x', 'avg_distance', 'headlines', 'avg_confidence_y',
       'representative_distance'],
      dtype='object')

In [39]:
topics_df = topics_df.rename(
    columns={
        "avg_confidence_x": "avg_confidence"
    }
)

topics_df = topics_df.drop(
    columns=[
        "avg_confidence_y",
        "headlines",
        "representative_distance"
    ],
    errors="ignore"
)

topics_df.head()

,cluster,topic_name,headline,keywords,headline_count,avg_confidence,avg_distance,headline_count,avg_distance
0,1521,Ceo 2016 2018,From Applied Materials Q2 Earnings Conference ...,"[earnings transcript, results earnings, transc...",36014,0.588888,0.521184,36014,0.521184
1,310,Declares Dividend Fund,Dundee Corporation Declares Quarterly First Pr...,"[declares, dividend, fund declares, trust decl...",26158,0.582308,0.537981,26158,0.537981
2,391,Buys Sells Management,"Greenlight Capital Buys Yahoo!, Trims Apple, S...","[buys, llc buys, sells, corp, llc, management,...",25802,0.580003,0.537210,25802,0.537210
3,2996,Estimates Revenues Beats,Agilent Q4 Profit Beats Views Agilent Beats Q3...,"[earnings, estimates, revenues, earnings reven...",23335,0.571469,0.583278,23335,0.583278
4,1604,Buy Value Investors,Why Agilent Technologies (A) Stock Might be a ...,"[stock, buy, value, value investors, investors...",21029,0.570697,0.581782,21029,0.581782


In [43]:
topics_df = topics_df.loc[:, ~topics_df.columns.duplicated()]

print(topics_df.columns.tolist())

['cluster', 'topic_name', 'headline', 'keywords', 'headline_count', 'avg_confidence', 'avg_distance']


In [44]:
from collections import Counter

Counter(topics_df.columns)

Counter({'cluster': 1,
         'topic_name': 1,
         'headline': 1,
         'keywords': 1,
         'headline_count': 1,
         'avg_confidence': 1,
         'avg_distance': 1})

In [45]:
print(topics_df.columns.tolist())

['cluster', 'topic_name', 'headline', 'keywords', 'headline_count', 'avg_confidence', 'avg_distance']


In [46]:
topics_df = topics_df[

    [

        "cluster",

        "topic_name",

        "headline",

        "keywords",

        "headline_count",

        "avg_confidence",

        "avg_distance"

    ]

].sort_values(

    "headline_count",

    ascending=False

).reset_index(

    drop=True

)

topics_df.head()

,cluster,topic_name,headline,keywords,headline_count,avg_confidence,avg_distance
0,1521,Ceo 2016 2018,From Applied Materials Q2 Earnings Conference ...,"[earnings transcript, results earnings, transc...",36014,0.588888,0.521184
1,310,Declares Dividend Fund,Dundee Corporation Declares Quarterly First Pr...,"[declares, dividend, fund declares, trust decl...",26158,0.582308,0.537981
2,391,Buys Sells Management,"Greenlight Capital Buys Yahoo!, Trims Apple, S...","[buys, llc buys, sells, corp, llc, management,...",25802,0.580003,0.537210
3,2996,Estimates Revenues Beats,Agilent Q4 Profit Beats Views Agilent Beats Q3...,"[earnings, estimates, revenues, earnings reven...",23335,0.571469,0.583278
4,1604,Buy Value Investors,Why Agilent Technologies (A) Stock Might be a ...,"[stock, buy, value, value investors, investors...",21029,0.570697,0.581782


## 15. Save Final Dataset

In [47]:
TOPIC_FILE = os.path.join(

    TOPIC_PATH,

    "cluster_topics.parquet"

)

topics_df.to_parquet(

    TOPIC_FILE,

    index=False

)

print("=" * 70)
print("Topic Dataset Saved Successfully")
print("=" * 70)

print(f"Output File : {TOPIC_FILE}")
print(f"Total Topics : {len(topics_df):,}")

Topic Dataset Saved Successfully
Output File : /content/drive/MyDrive/FinSight_AI/data/topics/cluster_topics.parquet
Total Topics : 2,913


## 16.Summary

In [48]:
summary = pd.DataFrame({

    "Metric":[

        "Clusters",

        "Vocabulary Size",

        "Top Keywords",

        "Output File"

    ],

    "Value":[

        len(topics_df),

        len(feature_names),

        TOP_K,

        os.path.basename(

            TOPIC_FILE

        )

    ]

})

summary

,Metric,Value
0,Clusters,2913
1,Vocabulary Size,5000
2,Top Keywords,15
3,Output File,cluster_topics.parquet
